In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
from pathlib import Path

DATA_DIR     = Path("/kaggle/input/datasets/ubitquitin/geolocation-geoguessr-images-50k/compressed_dataset")
OUTPUT_DIR   = Path("/kaggle/working/Geoguessr_Big_Four")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COUNTRIES    = ["United States", "Japan", "Brazil", "Russia"]
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 15
LR           = 1e-4
NUM_WORKERS  = 2
SEED         = 42

import torch, numpy as np
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Config ready!")
print(f"Data dir exists: {DATA_DIR.exists()}")

Config ready!
Data dir exists: True


In [6]:
# The dataset stores images under per-country folders.
# Walk the directory and collect (path, label) pairs for our 4 countries.

records = []
for country in COUNTRIES:
    country_dir = DATA_DIR / country
    if not country_dir.exists():
        print(f"[WARN] folder not found: {country_dir}")
        continue
    for img_path in country_dir.glob("*.jpg"):
        records.append({"path": str(img_path), "country": country})

df = pd.DataFrame(records)
print(df["country"].value_counts())
df.head()

country
United States    12014
Japan             3840
Brazil            2320
Russia            1761
Name: count, dtype: int64


,path,country
0,/kaggle/input/datasets/ubitquitin/geolocation-...,United States
1,/kaggle/input/datasets/ubitquitin/geolocation-...,United States
2,/kaggle/input/datasets/ubitquitin/geolocation-...,United States
3,/kaggle/input/datasets/ubitquitin/geolocation-...,United States
4,/kaggle/input/datasets/ubitquitin/geolocation-...,United States


In [7]:
from sklearn.model_selection import train_test_split

# Balance classes by capping at the smallest country's count
min_count = df["country"].value_counts().min()
df_balanced = (df.groupby("country")
                 .apply(lambda x: x.sample(min_count, random_state=SEED))
                 .reset_index(drop=True))

print("Balanced counts:")
print(df_balanced["country"].value_counts())

label2idx = {c: i for i, c in enumerate(COUNTRIES)}
idx2label = {i: c for c, i in label2idx.items()}
df_balanced["label"] = df_balanced["country"].map(label2idx)

train_df, temp_df = train_test_split(df_balanced, test_size=0.2, stratify=df_balanced["label"], random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

print(f"\nTrain: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

Balanced counts:
country
Brazil           1761
Japan            1761
Russia           1761
United States    1761
Name: count, dtype: int64

Train: 5635  Val: 704  Test: 705


/tmp/ipykernel_57/3778784314.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min_count, random_state=SEED))


In [8]:
class GeoDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(row["label"])


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = GeoDataset(train_df, train_tf)
val_ds   = GeoDataset(val_df,   val_tf)
test_ds  = GeoDataset(test_df,  val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Batches — Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}")

Batches — Train: 177  Val: 22  Test: 23


In [9]:
def build_model(num_classes=4):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Linear(256, num_classes),
    )
    return model

model = build_model(num_classes=len(COUNTRIES)).to(device)
print(model.fc)
print(f"\nTrainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=2048, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=4, bias=True)
)

Trainable params: 525,572


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.fc.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total

best_val_acc = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"Val loss {vl_loss:.4f} acc {vl_acc:.3f}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pt")
        print("  ✓ saved best model")

print(f"\nBest val accuracy: {best_val_acc:.3f}")

Epoch 01/15 | Train loss 1.3024 acc 0.420 | Val loss 1.2087 acc 0.537
  ✓ saved best model
Epoch 02/15 | Train loss 1.1396 acc 0.546 | Val loss 1.0912 acc 0.601
  ✓ saved best model
Epoch 03/15 | Train loss 1.0288 acc 0.591 | Val loss 0.9876 acc 0.632
  ✓ saved best model
Epoch 04/15 | Train loss 0.9713 acc 0.610 | Val loss 0.9347 acc 0.670
  ✓ saved best model
Epoch 05/15 | Train loss 0.9317 acc 0.634 | Val loss 0.9316 acc 0.641
Epoch 06/15 | Train loss 0.8977 acc 0.655 | Val loss 0.8910 acc 0.672
  ✓ saved best model
Epoch 07/15 | Train loss 0.8917 acc 0.648 | Val loss 0.8544 acc 0.699
  ✓ saved best model
Epoch 08/15 | Train loss 0.8646 acc 0.666 | Val loss 0.8499 acc 0.685
Epoch 09/15 | Train loss 0.8457 acc 0.671 | Val loss 0.8277 acc 0.702
  ✓ saved best model
Epoch 10/15 | Train loss 0.8313 acc 0.679 | Val loss 0.8318 acc 0.700
Epoch 11/15 | Train loss 0.8384 acc 0.678 | Val loss 0.8087 acc 0.713
  ✓ saved best model
Epoch 12/15 | Train loss 0.8214 acc 0.678 | Val loss 0.8140 ac

In [11]:
# Reset to best original model
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pt"))

# Unfreeze ONLY layer4 (the final feature layer — safer than layer3+4)
for name, param in model.named_parameters():
    param.requires_grad = "layer4" in name or "fc" in name

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

optimizer_ft = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-6,  # much smaller LR
    weight_decay=1e-4
)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=10)

best_val_acc_ft = 0.0
for epoch in range(1, 11):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader, train=False)
    optimizer_ft.step()
    scheduler_ft.step()

    print(f"FT Epoch {epoch:02d}/10 | "
          f"Train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"Val loss {vl_loss:.4f} acc {vl_acc:.3f}")

    if vl_acc > best_val_acc_ft:
        best_val_acc_ft = vl_acc
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model_ft.pt")
        print("  ✓ saved best fine-tuned model")

print(f"\nBest fine-tuned val accuracy: {best_val_acc_ft:.3f}")

Trainable params: 15,490,308
FT Epoch 01/10 | Train loss 0.8313 acc 0.668 | Val loss 0.8253 acc 0.710
  ✓ saved best fine-tuned model
FT Epoch 02/10 | Train loss 0.8216 acc 0.684 | Val loss 0.8115 acc 0.717
  ✓ saved best fine-tuned model
FT Epoch 03/10 | Train loss 0.8241 acc 0.679 | Val loss 0.8092 acc 0.714
FT Epoch 04/10 | Train loss 0.8058 acc 0.691 | Val loss 0.8148 acc 0.727
  ✓ saved best fine-tuned model
FT Epoch 05/10 | Train loss 0.8278 acc 0.681 | Val loss 0.8159 acc 0.716
FT Epoch 06/10 | Train loss 0.8230 acc 0.677 | Val loss 0.8082 acc 0.706
FT Epoch 07/10 | Train loss 0.8315 acc 0.676 | Val loss 0.8412 acc 0.705
FT Epoch 08/10 | Train loss 0.8153 acc 0.684 | Val loss 0.8195 acc 0.713
FT Epoch 09/10 | Train loss 0.8167 acc 0.681 | Val loss 0.8159 acc 0.703
FT Epoch 10/10 | Train loss 0.8105 acc 0.687 | Val loss 0.8175 acc 0.714

Best fine-tuned val accuracy: 0.727


In [12]:
import shutil
shutil.copy(
    '/kaggle/working/Geoguessr_Big_Four/best_model_ft.pt',
    '/kaggle/working/best_model_ft.pt'
)

'/kaggle/working/best_model_ft.pt'

In [13]:
from IPython.display import FileLink
FileLink('/kaggle/working/best_model_ft.pt')

/kaggle/working/best_model_ft.pt